In [1]:
# ============================================================
# PHASE 7 — DATA SPLITS
# Load preprocessed learning units and verify split grouping
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

CRIC_MANIFEST = PROCESSED_ROOT / "cric_learning_units.csv"
RIVA_MANIFEST = PROCESSED_ROOT / "riva_learning_units.csv"

# ------------------------------------------------------------
# Load preprocessed datasets
# ------------------------------------------------------------
cric_df = pd.read_csv(CRIC_MANIFEST)
riva_df = pd.read_csv(RIVA_MANIFEST)

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------
print("PHASE 7 — DATA SPLITS")
print("=" * 50)

print("\nCRIC")
print(f"Learning units: {len(cric_df):,}")
print(f"Parent images:  {cric_df['image_filename'].nunique():,}")

print("\nRIVA")
print(f"Learning units: {len(riva_df):,}")
print(f"Parent images:  {riva_df['image_filename'].nunique():,}")

print("\nBinary labels")
print("CRIC:")
print(cric_df["binary_label"].value_counts().sort_index())

print("\nRIVA:")
print(riva_df["binary_label"].value_counts().sort_index())

# ------------------------------------------------------------
# Confirm parent-image grouping is available
# ------------------------------------------------------------
assert cric_df["image_filename"].notna().all()
assert riva_df["image_filename"].notna().all()

print("\nParent-image grouping: AVAILABLE")
print("Ready to create leakage-safe splits.")

PHASE 7 — DATA SPLITS

CRIC
Learning units: 11,534
Parent images:  400

RIVA
Learning units: 15,949
Parent images:  959

Binary labels
CRIC:
binary_label
0    6779
1    4755
Name: count, dtype: int64

RIVA:
binary_label
0    10517
1     5432
Name: count, dtype: int64

Parent-image grouping: AVAILABLE
Ready to create leakage-safe splits.


In [2]:
# ============================================================
# PHASE 7 — LEAKAGE-SAFE TRAIN / VALIDATION / TEST SPLITS
# ============================================================

from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

def create_image_level_split(df, dataset_name):
    # One row per parent image
    image_labels = (
        df.groupby("image_filename")["binary_label"]
        .max()
        .reset_index()
    )

    # 70% train, 30% temporary
    train_images, temp_images = train_test_split(
        image_labels,
        test_size=0.30,
        random_state=RANDOM_SEED,
        stratify=image_labels["binary_label"]
    )

    # Split remaining 30% into 15% validation + 15% test
    val_images, test_images = train_test_split(
        temp_images,
        test_size=0.50,
        random_state=RANDOM_SEED,
        stratify=temp_images["binary_label"]
    )

    train_set = set(train_images["image_filename"])
    val_set = set(val_images["image_filename"])
    test_set = set(test_images["image_filename"])

    # Assign split to every cell
    result = df.copy()

    result["split"] = result["image_filename"].map(
        lambda x:
            "train" if x in train_set
            else "val" if x in val_set
            else "test"
    )

    print(f"\n{dataset_name}")
    print("-" * 40)
    print(
        f"Images: "
        f"train={len(train_set)}, "
        f"val={len(val_set)}, "
        f"test={len(test_set)}"
    )

    print("\nLearning units:")
    print(result["split"].value_counts())

    print("\nBinary labels by split:")
    print(
        pd.crosstab(
            result["split"],
            result["binary_label"]
        ).sort_index()
    )

    # Leakage check
    train_val = train_set & val_set
    train_test = train_set & test_set
    val_test = val_set & test_set

    assert not train_val
    assert not train_test
    assert not val_test

    print("\nImage leakage check: PASSED")

    return result


cric_splits_df = create_image_level_split(
    cric_df,
    "CRIC"
)

riva_splits_df = create_image_level_split(
    riva_df,
    "RIVA"
)


CRIC
----------------------------------------
Images: train=280, val=60, test=60

Learning units:
split
train    8274
test     1721
val      1539
Name: count, dtype: int64

Binary labels by split:
binary_label     0     1
split                   
test          1108   613
train         4736  3538
val            935   604

Image leakage check: PASSED

RIVA
----------------------------------------
Images: train=671, val=144, test=144

Learning units:
split
train    11041
val       2483
test      2425
Name: count, dtype: int64

Binary labels by split:
binary_label     0     1
split                   
test          1668   757
train         7271  3770
val           1578   905

Image leakage check: PASSED


In [3]:
# PHASE 7 — RIVA SLIDE-LEVEL SPLIT
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

# Extract RIVA slide ID:
# filename format = category_sample-number_mini-patch-number.png
# Example: HSIL_1_1.png -> slide = HSIL_1
def get_riva_slide_id(filename):
    stem = Path(filename).stem
    parts = stem.split("_")
    return "_".join(parts[:-1])   # remove mini-patch number


# Create slide-level grouping
riva_df["slide_id"] = riva_df["image_filename"].apply(get_riva_slide_id)

# One representative label per slide for stratification
slide_labels = (
    riva_df.groupby("slide_id")["binary_label"]
    .max()
    .reset_index()
)

# 70% train, 30% temporary
train_slides, temp_slides = train_test_split(
    slide_labels,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=slide_labels["binary_label"]
)

# Split temporary 50/50 -> 15% validation, 15% test
val_slides, test_slides = train_test_split(
    temp_slides,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp_slides["binary_label"]
)

train_set = set(train_slides["slide_id"])
val_set = set(val_slides["slide_id"])
test_set = set(test_slides["slide_id"])

# Assign split to every RIVA learning unit
riva_splits_df = riva_df.copy()

riva_splits_df["split"] = riva_splits_df["slide_id"].map(
    lambda x: (
        "train" if x in train_set
        else "val" if x in val_set
        else "test"
    )
)

# Verification
print("RIVA — SLIDE-LEVEL SPLIT")
print("=" * 50)

print(f"Slides: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}")
print(f"Images: {riva_splits_df['image_filename'].nunique():,}")

print("\nLearning units:")
print(riva_splits_df["split"].value_counts())

print("\nBinary labels by split:")
print(
    pd.crosstab(
        riva_splits_df["split"],
        riva_splits_df["binary_label"]
    ).sort_index()
)

# Check slide leakage
train_val = train_set & val_set
train_test = train_set & test_set
val_test = val_set & test_set

assert len(train_val) == 0
assert len(train_test) == 0
assert len(val_test) == 0

# Check every image belongs to exactly one split
image_split_counts = riva_splits_df.groupby("image_filename")["split"].nunique()
assert image_split_counts.max() == 1

print("\nSlide leakage check: PASSED")
print("Image leakage check: PASSED")

RIVA — SLIDE-LEVEL SPLIT
Slides: train=77, val=17, test=17
Images: 959

Learning units:
split
train    10757
val       2640
test      2552
Name: count, dtype: int64

Binary labels by split:
binary_label     0     1
split                   
test          1634   918
train         7279  3478
val           1604  1036

Slide leakage check: PASSED
Image leakage check: PASSED


In [4]:
# PHASE 7 — SAVE SPLIT ASSIGNMENTS

SPLITS_ROOT = PROJECT_ROOT / "data" / "splits"
SPLITS_ROOT.mkdir(parents=True, exist_ok=True)

# Save CRIC and RIVA split assignments
cric_splits_df.to_csv(SPLITS_ROOT / "cric_splits.csv", index=False)
riva_splits_df.to_csv(SPLITS_ROOT / "riva_splits.csv", index=False)

print("Splits saved successfully:")
print(SPLITS_ROOT / "cric_splits.csv")
print(SPLITS_ROOT / "riva_splits.csv")

# Confirm
print("\nFiles:")
for f in SPLITS_ROOT.glob("*.csv"):
    print(f"  ✓ {f.name}")

Splits saved successfully:
c:\Users\nanda\Documents\cervical-semi-sl\data\splits\cric_splits.csv
c:\Users\nanda\Documents\cervical-semi-sl\data\splits\riva_splits.csv

Files:
  ✓ cric_splits.csv
  ✓ riva_splits.csv
